In [1]:
import sys
import cv2
import time
from matplotlib import pyplot as plt
from tqdm import tqdm, trange
import numpy as np
import pandas as pd
import pickle
import random
import copy
import json

import os
import shutil
from PIL import Image, ImageDraw

import torch
from torch.utils.data import Dataset
from torchvision.transforms.functional import to_tensor, normalize
from torchvision import transforms

from coco.dataset import COCODataset, COCODatasetRandom
from utils import *
sys.path.append("..")

In [2]:
test_annotations = '../datasets/COCO18_dset_for_CRT_training/coco18_test_deepgaze1.json'
test_imagedir = '../datasets/COCO18_dset_for_CRT_training/coco18_test_deepgaze_resize/'

In [3]:
import json
import pickle

with open(test_annotations, 'rb') as file:
    train_metadata = json.load(file)
    
categories_id_to_name = {}

# _____ CREATE A LOOKUP TABLE FOR CATEGORY ID TO CATEGORY NAME _____
for info in train_metadata['categories']:
    categories_id_to_name[info['id']] = info['name']
# _____ CREATE A LOOKUP TABLE FOR CATEGORY ID TO CATEGORY NAME _____
    
    
    
    
    
# _____ CREATE A LOOKUP TABLE FOR INDEXES IN THE SAME CATEGORIES _____
category_idx_dict = {}

for i in range(len(train_metadata['annotations'])):
    
    info = train_metadata['annotations'][i]
    
    name = categories_id_to_name[info['category_id']]
    
    if name not in category_idx_dict:
        category_idx_dict[name] = []
        
    category_idx_dict[name].append(i)
# _____ CREATE A LOOKUP TABLE FOR INDEXES IN THE SAME CATEGORIES _____



with open('../datasets/COCO18_dset_for_CRT_training/coco18_idx_test612.pkl', 'wb') as file:
    pickle.dump(category_idx_dict, file, protocol=pickle.HIGHEST_PROTOCOL)

In [4]:
category_dic_dir = '../datasets/COCO18_dset_for_CRT_training/coco18_idx_test612.pkl'
image_list = os.listdir(test_imagedir)
coco_dataset = COCODatasetRandom(test_annotations, test_imagedir, image_size =(224,224), category_dic_dir = category_dic_dir, normalize_means=[0.485, 0.456, 0.406], normalize_stds=[0.229, 0.224, 0.225])

-------------------------------
Annotation Counts
-------------------------------
chair                        43
fork                         41
sink                         51
tv                           50
bowl                         26
car                          20
clock                        23
cup                          49
keyboard                     33
knife                        24
laptop                       23
mouse                        19
oven                         19
potted plant                 28
toilet                       29
bottle                       30
stop sign                    25
microwave                    27
Total                       560
-------------------------------



In [1]:
context_dir = "../datasets/SCEGRAM/SCEGRAM/01scenes/01object_present"
target_dir  = "../datasets/SCEGRAM/SCEGRAM/invariant_objects"
# target_dir  = "../datasets/SCEGRAM/SCEGRAM/02objects"
info_dir    = "../datasets/SCEGRAM/SCEGRAM/SCEGRAM_Database_scenes_objects.xlsx"

context_size, target_size = (320, 512), (128, 128)
scegram_dataset = SCEGRAM(info_dir, context_dir, target_dir, context_size, target_size)